# Recurrent networks: RNN, GRU and LSTM

A recurrent network reads a sequence one step at a time and carries a hidden state forward, so the last step can depend on the first. That is also where it breaks. Unrolled over $T$ steps, it is a $T$-layer network with the same weights in every layer, and the unstable gradients of notebook 05 come straight back. This notebook builds a vanilla RNN in NumPy and measures how its gradient dies with distance. Then it trains PyTorch's `RNN`, `GRU` and `LSTM` on a memory task and reports what happened, including where the textbook story did not show.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import get_device, set_seed, setup_keras

setup_keras("torch")
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from common.sequences import make_memory_task, one_hot

SEED = 42
set_seed(SEED)
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)
device = get_device()
torch.backends.cudnn.deterministic = True  # same seed, same numbers on a GPU
print("device:", device)

## 1. Intuition

The task is delayed recall. Each sequence shows one of four symbols at step 0, then blanks (symbol 4) for the rest. The label is the first symbol. A model must hold it in its hidden state for $T-1$ steps. Nothing else in the sequence helps, which isolates memory from everything else.

Sequences are generated by `make_memory_task` in `common/sequences.py`, seeded and tested.

## 2. Maths and cost function

The vanilla RNN updates $\mathbf h_t=\tanh(W_x\mathbf x_t+W_h\mathbf h_{t-1}+\mathbf b)$ and reads the answer from the last state. The cost is cross-entropy on that read-out. Backprop through time sends the gradient back one step at a time:

$$\frac{\partial L}{\partial \mathbf h_{t-1}}=W_h^\top\,\mathrm{diag}\big(1-\mathbf h_t^2\big)\,\frac{\partial L}{\partial \mathbf h_t}$$

Over $k$ steps that is $k$ copies of the same $W_h^\top$ and the tanh slope, at most 1. If the largest singular value of $W_h$ is below 1, the gradient shrinks geometrically. Above 1, the weights push it up while the tanh slope pushes it down. This is the product from notebook 05, with one matrix repeated instead of $L$ different ones.

GRU and LSTM add gates. The LSTM keeps a separate cell state that is updated by addition, $\mathbf c_t=\mathbf f_t\odot\mathbf c_{t-1}+\mathbf i_t\odot\tilde{\mathbf c}_t$. When the forget gate $\mathbf f_t$ is near 1, the gradient passes along the cell state almost unchanged.

## 3. From scratch

A NumPy RNN with a forward pass and a backward pass through time. The recurrent matrix is orthogonal times a scale, so the scale is its largest singular value. We measure the gradient norm at every step for scales 0.5, 1.0 and 1.5 over $T=40$ steps.

In [ ]:
x_seq, y_seq = make_memory_task(4, seq_len=8, n_symbols=4, seed=SEED)  # 4 is the blank
print("sequences:\n", x_seq, "\nlabels (the first symbol):", y_seq)

At scale 0.5 the gradient at step 1 is $5.5\times10^{-15}$ of the gradient at step 40. At 1.0, where the weights alone would preserve it, tanh still cuts it to $2\times10^{-3}$. At 1.5 it does not vanish (the ratio is 3.2), because the weights now outgrow the tanh slope. That is the unstable-gradient fork from notebook 05, along the time axis. Gradient clipping from that notebook is the standard guard for the exploding side.

## 4. Library version

PyTorch's `nn.RNN`, `nn.GRU` and `nn.LSTM` share one interface, so a single wrapper covers all three. First, the same measurement on the real layers at their default initialisation: how much does the loss feel the input at step $t$? The model reads only the last step, so the gradient to input $t$ is the signal that survives $T-t$ steps.

In [ ]:
class NumpyRNN:
    """h_t = tanh(Wx x_t + Wh h_{t-1} + b), then a linear read-out from the last hidden state."""

    def __init__(self, n_in: int, n_hidden: int, n_out: int, recurrent_scale: float, seed: int = 0):
        g = np.random.default_rng(seed)
        self.Wx = g.normal(0, 1 / np.sqrt(n_in), (n_in, n_hidden))
        # Orthogonal-ish recurrent matrix, rescaled so its largest singular value is recurrent_scale.
        q, _ = np.linalg.qr(g.normal(size=(n_hidden, n_hidden)))
        self.Wh = q * recurrent_scale
        self.b = np.zeros(n_hidden)
        self.Wo = g.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, n_out))

    def forward(self, x: np.ndarray) -> tuple[np.ndarray, list[np.ndarray]]:
        h = np.zeros((len(x), self.b.size))
        hs = [h]
        for t in range(x.shape[1]):
            h = np.tanh(x[:, t] @ self.Wx + h @ self.Wh + self.b)
            hs.append(h)
        return hs[-1] @ self.Wo, hs

    def grad_norms_over_time(self, x: np.ndarray, y: np.ndarray) -> np.ndarray:
        """Norm of dLoss/dh_t for every step t, by backprop through time from the last step."""
        logits, hs = self.forward(x)
        p = np.exp(logits - logits.max(1, keepdims=True))
        p /= p.sum(1, keepdims=True)
        p[np.arange(len(y)), y] -= 1  # softmax cross-entropy gradient
        dh = (p / len(y)) @ self.Wo.T
        norms = [np.linalg.norm(dh)]
        for t in range(x.shape[1], 1, -1):
            dh = (dh * (1 - hs[t] ** 2)) @ self.Wh.T  # one step back: tanh slope, then Wh
            norms.append(np.linalg.norm(dh))
        return np.array(norms[::-1])  # index 0 = the first time step


T = 40
x_np, y_np = make_memory_task(64, T, n_symbols=4, seed=SEED)
x_oh = one_hot(x_np, 5)
for scale in (0.5, 1.0, 1.5):
    rnn = NumpyRNN(5, 32, 4, recurrent_scale=scale, seed=SEED)
    g = rnn.grad_norms_over_time(x_oh, y_np)
    plt.semilogy(range(1, T + 1), g, label=f"Wh scale {scale}")
    print(f"scale {scale}: gradient at step 1 / step {T} = {g[0] / g[-1]:.1e}")
plt.xlabel("time step")
plt.ylabel("norm of dLoss/dh_t")
plt.legend()
plt.show()

All three vanish at initialisation, by 6 to 7 orders of magnitude across 30 steps: $5.8\times10^{-8}$ (RNN), $5.1\times10^{-7}$ (GRU) and $7.0\times10^{-7}$ (LSTM). The gates do not fix this on their own. They give the network a path it can learn to use. Whether it finds that path is a training question, so now we train. Three seeds per model, 25 epochs of Adam at 0.01, 1,200 training sequences.

In [ ]:
class Recurrent(nn.Module):
    def __init__(self, kind: str, n_in: int = 5, n_hidden: int = 32, n_out: int = 4) -> None:
        super().__init__()
        cell = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[kind]
        self.rnn = cell(n_in, n_hidden, batch_first=True)
        self.out = nn.Linear(n_hidden, n_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq, _ = self.rnn(x)  # (batch, time, hidden)
        return self.out(seq[:, -1])  # read the last step only


T = 30
xs, ys = make_memory_task(256, T, seed=SEED)
Xg, Yg = torch.from_numpy(one_hot(xs, 5)).to(device), torch.from_numpy(ys).to(device)
for kind in ("rnn", "gru", "lstm"):
    set_seed(SEED)
    net = Recurrent(kind).to(device)
    Xg.requires_grad_(True)
    Xg.grad = None
    nn.functional.cross_entropy(net(Xg), Yg).backward()
    g = Xg.grad.norm(dim=(0, 2)).cpu().numpy()  # how much the loss feels each input step
    plt.semilogy(range(1, T + 1), g, label=kind)
    print(f"{kind:>4}: gradient at step 1 / step {T} = {g[0] / g[-1]:.1e}")
plt.xlabel("time step")
plt.ylabel("norm of dLoss/dx_t")
plt.legend()
plt.show()

This is not the textbook result. The vanilla RNN and the GRU solve length 30 on every seed. The LSTM stays at 22 to 23% accuracy, which is chance (25%), on all three. I did not run a longer budget or other settings to find out whether it escapes.

## 5. Tuning

Sequence length is the dial. At length 10 all three should manage.

In [ ]:
def fit_memory(kind: str, seq_len: int, seed: int, epochs: int = 25) -> float:
    """Train on delayed recall and return test accuracy."""
    xs, ys = make_memory_task(1600, seq_len, seed=seed)
    X, Y = torch.from_numpy(one_hot(xs, 5)).to(device), torch.from_numpy(ys).to(device)
    n_tr = 1200
    set_seed(seed)
    model = Recurrent(kind).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        order = torch.randperm(n_tr, generator=g)
        for i in range(0, n_tr, 50):
            idx = order[i : i + 50].to(device)
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[idx]), Y[idx]).backward()
            opt.step()
    with torch.no_grad():
        return (model(X[n_tr:]).argmax(1) == Y[n_tr:]).float().mean().item()


SEEDS = (0, 1, 2)
for kind in ("rnn", "gru", "lstm"):
    accs = [fit_memory(kind, 30, s) for s in SEEDS]
    print(f"{kind:>4}, length 30, three seeds: {np.round(accs, 2)}, mean {np.mean(accs):.2f}")

The RNN and GRU score 100% on every seed and the LSTM on two of three (50% on the third). On a CPU run the LSTM also scores 100% on all three, so this single odd result is seed and device noise. Short tasks hide the differences, and so do easy ones: this task has blank inputs, so the hidden state is not disturbed by noise.

## 6. What breaks

Length 60, two seeds each.

In [ ]:
for kind in ("rnn", "gru", "lstm"):
    accs = [fit_memory(kind, 10, s) for s in SEEDS]
    print(f"{kind:>4}, length 10, three seeds: {np.round(accs, 2)}, mean {np.mean(accs):.2f}")

The RNN gets 100% and 79%, so its gradient was still usable, probably because Adam rescales each parameter's step and makes up for tiny gradients. The GRU and LSTM stay at chance on both seeds. Plain SGD is less forgiving and would likely show the vanishing gradient directly, but I have not tested it here. What the run does support: the measured gradient decays by many orders of magnitude, the optimiser and the task decide how much that matters, and results from one seed on a sequence task should not be trusted.

In [ ]:
for kind in ("rnn", "gru", "lstm"):
    accs = [fit_memory(kind, 60, s) for s in SEEDS[:2]]
    print(f"{kind:>4}, length 60, two seeds: {np.round(accs, 2)}")

The cost of the gates is parameters. At hidden size 32 the RNN has 1,248, the GRU 3,744 (three times) and the LSTM 4,992 (four times), because each gate is another set of weights.

In [ ]:
for kind in ("rnn", "gru", "lstm"):
    counts = [
        sum(p.numel() for p in Recurrent(kind, n_hidden=h).rnn.parameters()) for h in (16, 32, 64)
    ]
    print(f"{kind:>4} parameters at hidden size 16, 32, 64: {counts}")

The gradient decay was measured and the ranking of the models was not what the textbook says. On a harder task, does the LSTM beat the RNN over a few seeds, and by how much? The language models in section 06 are where that matters.